# Quick Draw Demo - Jupyter Version

Interactive drawing canvas for testing your trained model.

## Setup
1. Place your model file in the same folder
2. Run cells in order
3. Draw on the canvas and click **Recognize**

In [11]:
# Install if needed (run once)
# !pip install torch torchvision timm ipycanvas ipywidgets

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import timm
from ipycanvas import Canvas, hold_canvas
from ipywidgets import Button, HBox, VBox, Output, Dropdown, HTML
from IPython.display import display, clear_output

In [22]:
# ============================================
# CONFIGURATION - CHANGE MODEL HERE
# ============================================

MODELS = {
    "old": {
        "arch": "mobilenetv3_large_100",
        "size": 64,
        "path": "best_model.pt",
        "name": "MobileNetV3 (64x64)"
    },
    "new": {
        "arch": "efficientnet_b2",
        "size": 96,
        "path": "best_model_new.pt",
        "name": "EfficientNet-B2 (96x96)"
    }
}

NUM_CLASSES = 340
CANVAS_SIZE = 400
LINE_WIDTH = 10

In [ ]:
# ============================================
# ALL 340 CLASSES
# ============================================

CLASSES = ['airplane', 'alarm clock', 'ambulance', 'angel', 'animal migration', 'ant', 'anvil', 'apple', 'arm', 'asparagus', 'axe', 'backpack', 
'banana', 'bandage', 'barn', 'baseball', 'baseball bat', 'basket', 'basketball', 'bat', 'bathtub', 'beach', 'bear', 'beard', 'bed', 'bee', 'belt', 
'bench', 'bicycle', 'binoculars', 'bird', 'birthday cake', 'blackberry', 'blueberry', 'book', 'boomerang', 'bottlecap', 'bowtie', 'bracelet', 'brain', 
'bread', 'bridge', 'broccoli', 'broom', 'bucket', 'bulldozer', 'bus', 'bush', 'butterfly', 'cactus', 'cake', 'calculator', 'calendar', 'camel', 'camera', 
'camouflage', 'campfire', 'candle', 'cannon', 'canoe', 'car', 'carrot', 'castle', 'cat', 'ceiling fan', 'cell phone', 'cello', 'chair', 'chandelier', 'church', 
'circle', 'clarinet', 'clock', 'cloud', 'coffee cup', 'compass', 'computer', 'cookie', 'cooler', 'couch', 'cow', 'crab', 'crayon', 'crocodile', 'crown', 'cruise ship', 
'cup', 'diamond', 'dishwasher', 'diving board', 'dog', 'dolphin', 'donut', 'door', 'dragon', 'dresser', 'drill', 'drums', 'duck', 'dumbbell', 'ear', 'elbow', 'elephant', 
'envelope', 'eraser', 'eye', 'eyeglasses', 'face', 'fan', 'feather', 'fence', 'finger', 'fire hydrant', 'fireplace', 'firetruck', 'fish', 'flamingo', 'flashlight', 'flip flops', 
'floor lamp', 'flower', 'flying saucer', 'foot', 'fork', 'frog', 'frying pan', 'garden', 'garden hose', 'giraffe', 'goatee', 'golf club', 'grapes', 'grass', 'guitar', 'hamburger', 
'hammer', 'hand', 'harp', 'hat', 'headphones', 'hedgehog', 'helicopter', 'helmet', 'hexagon', 'hockey puck', 'hockey stick', 'horse', 'hospital', 'hot air balloon', 'hot dog', 'hot tub', 
'hourglass', 'house', 'house plant', 'hurricane', 'ice cream', 'jacket', 'jail', 'kangaroo', 'key', 'keyboard', 'knee', 'ladder', 'lantern', 'laptop', 'leaf', 'leg', 'light bulb', 'lighthouse', 
'lightning', 'line', 'lion', 'lipstick', 'lobster', 'lollipop', 'mailbox', 'map', 'marker', 'matches', 'megaphone', 'mermaid', 'microphone', 'microwave', 'monkey', 'moon', 'mosquito', 'motorbike', 
'mountain', 'mouse', 'moustache', 'mouth', 'mug', 'mushroom', 'nail', 'necklace', 'nose', 'ocean', 'octagon', 'octopus', 'onion', 'oven', 'owl', 'paint can', 'paintbrush', 'palm tree', 'panda', 
'pants', 'paper clip', 'parachute', 'parrot', 'passport', 'peanut', 'pear', 'peas', 'pencil', 'penguin', 'piano', 'pickup truck', 'picture frame', 'pig', 'pillow', 'pineapple', 'pizza', 'pliers', 
'police car', 'pond', 'pool', 'popsicle', 'postcard', 'potato', 'power outlet', 'purse', 'rabbit', 'raccoon', 'radio', 'rain', 'rainbow', 'rake', 'remote control', 'rhinoceros', 'river', 
'roller coaster', 'rollerskates', 'sailboat', 'sandwich', 'saw', 'saxophone', 'school bus', 'scissors', 'scorpion', 'screwdriver', 'sea turtle', 'see saw', 'shark', 'sheep', 'shoe', 'shorts', 
'shovel', 'sink', 'skateboard', 'skull', 'skyscraper', 'sleeping bag', 'smiley face', 'snail', 'snake', 'snorkel', 'snowflake', 'snowman', 'soccer ball', 'sock', 'speedboat', 'spider', 'spoon', 
'spreadsheet', 'square', 'squiggle', 'squirrel', 'stairs', 'star', 'steak', 'stereo', 'stethoscope', 'stitches', 'stop sign', 'stove', 'strawberry', 'streetlight', 'string bean', 'submarine', 
'suitcase', 'sun', 'swan', 'sweater', 'swing set', 'sword', 't-shirt', 'table', 'teapot', 'teddy-bear', 'telephone', 'television', 'tennis racquet', 'tent', 'The Eiffel Tower', 
'The Great Wall of China', 'The Mona Lisa', 'tiger', 'toaster', 'toe', 'toilet', 'tooth', 'toothbrush', 'toothpaste', 'tornado', 'tractor', 'traffic light', 'train', 'tree', 'triangle', 
'trombone', 'truck', 'trumpet', 'umbrella', 'underwear', 'van', 'vase', 'violin', 'washing machine', 'watermelon', 'waterslide', 'whale', 'wheel', 'windmill', 'wine bottle', 'wine glass', 
'wristwatch', 'yoga', 'zebra', 'zigzag']
 

ID2CLASS = {i: c.replace(' ', '_') for i, c in enumerate(CLASSES)}
print(f"✓ Loaded {len(CLASSES)} classes")

✓ Loaded 340 classes


In [24]:
# ============================================
# LOAD MODEL
# ============================================

def load_model(model_key):
    config = MODELS[model_key]
    
    # Device
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    
    # Create model
    model = timm.create_model(
        config["arch"],
        pretrained=False,
        in_chans=1,
        num_classes=NUM_CLASSES
    )
    
    # Load weights
    if Path(config["path"]).exists():
        state_dict = torch.load(config["path"], map_location=device, weights_only=True)

        # Strip 'backbone.' prefix if present
        new_state_dict = {}
        for k, v in state_dict.items():
            new_key = k.replace("backbone.", "") if k.startswith("backbone.") else k
            new_state_dict[new_key] = v
        model.load_state_dict(new_state_dict)
        print(f"✓ Loaded: {config['path']}")
    else:
        print(f"⚠ Model not found: {config['path']}")
        return None, None, None
    
    model.to(device)
    model.eval()
    
    print(f"✓ Model: {config['name']}")
    print(f"✓ Device: {device}")
    
    return model, device, config

# Load default model
current_model_key = "new"  # Change to "old" for old model
model, device, config = load_model(current_model_key)

✓ Loaded: best_model_new.pt
✓ Model: EfficientNet-B2 (96x96)
✓ Device: mps


In [25]:
# ============================================
# PREDICTION FUNCTION
# ============================================

@torch.no_grad()
def predict(canvas_data, model, config, device):
    """Get predictions from canvas."""
    import cv2
    
    # Resize
    img = cv2.resize(canvas_data, (config["size"], config["size"]), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32)
    img = (img / 127.5) - 1.0
    tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).to(device)
    
    # Predict
    outputs = model(tensor)
    probs = torch.softmax(outputs, dim=1)
    top_probs, top_indices = probs.topk(5, dim=1)
    
    results = []
    for prob, idx in zip(top_probs[0], top_indices[0]):
        results.append((ID2CLASS[idx.item()], prob.item()))
    
    return results

In [27]:
# ============================================
# INTERACTIVE CANVAS WITH TIME-COLORED STROKES
# ============================================

from ipycanvas import Canvas, hold_canvas
from ipywidgets import Button, HBox, VBox, Output, HTML, Layout
from IPython.display import display, clear_output
import numpy as np
import cv2

# Settings
SIZE = 280
LINE_WIDTH = 6

# Create canvas
canvas = Canvas(width=SIZE, height=SIZE, sync_image_data=True, layout=Layout(width=f'{SIZE}px', height=f'{SIZE}px'))
canvas.fill_style = "black"
canvas.fill_rect(0, 0, SIZE, SIZE)
canvas.line_width = LINE_WIDTH
canvas.line_cap = "round"

output = Output()

# Track strokes with different colors (like training data)
stroke_count = 0
drawing = False
last_x, last_y = 0, 0

# Colors: first stroke = brightest, fades with each stroke
def get_stroke_color(stroke_num):
    # Match training: color = 255 - min(t, 10) * 13
    intensity = 255 - min(stroke_num, 10) * 13
    return f'rgb({intensity},{intensity},{intensity})'

def on_mouse_down(x, y):
    global drawing, last_x, last_y, stroke_count
    drawing = True
    last_x, last_y = x, y
    # Set color for this stroke
    canvas.stroke_style = get_stroke_color(stroke_count)

def on_mouse_move(x, y):
    global drawing, last_x, last_y
    if drawing:
        canvas.stroke_line(last_x, last_y, x, y)
        last_x, last_y = x, y

def on_mouse_up(x, y):
    global drawing, stroke_count
    drawing = False
    stroke_count += 1  # Next stroke will be darker

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

def clear_clicked(btn):
    global stroke_count
    stroke_count = 0
    canvas.fill_style = "black"
    canvas.fill_rect(0, 0, SIZE, SIZE)
    with output:
        clear_output()
        print("Cleared! (stroke count reset)")

def recognize_clicked(btn):
    global model, device, current_model_key
    
    if model is None:
        with output:
            clear_output()
            print("❌ Model not loaded! Run load_model() first")
        return
    
    # Get canvas data
    data = canvas.get_image_data(0, 0, SIZE, SIZE)
    img_array = np.array(data).reshape(SIZE, SIZE, 4)
    gray = img_array[:, :, 0]  # Red channel has our grayscale
    
    # Predict
    size = MODELS[current_model_key]["size"]
    results = predict(gray, model, {"size": size}, device)
    
    with output:
        clear_output()
        print(f"Strokes drawn: {stroke_count}")
        print("="*40)
        print("PREDICTIONS")
        print("="*40)
        for i, (name, prob) in enumerate(results):
            bar = "█" * int(prob * 30)
            print(f"{i+1}. {name.replace('_', ' '):<25} {prob*100:5.1f}% {bar}")

clear_btn = Button(description="Clear", button_style="warning")
clear_btn.on_click(clear_clicked)

recognize_btn = Button(description="Recognize", button_style="success")
recognize_btn.on_click(recognize_clicked)

buttons = HBox([recognize_btn, clear_btn])
title = HTML("<h3>🎨 Quick Draw</h3><p>First strokes = bright, later strokes = darker (like training data)</p>")

display(VBox([title, canvas, buttons, output], layout=Layout(width=f'{SIZE+20}px')))

In [37]:
# ============================================
# INTERACTIVE CANVAS
# ============================================

# Create canvas
canvas = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE)
canvas.fill_style = "black"
canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
canvas.stroke_style = "white"
canvas.line_width = LINE_WIDTH
canvas.line_cap = "round"

# Output for results
output = Output()

# Drawing state
drawing = False
last_x, last_y = 0, 0

def on_mouse_down(x, y):
    global drawing, last_x, last_y
    drawing = True
    last_x, last_y = x, y

def on_mouse_move(x, y):
    global drawing, last_x, last_y
    if drawing:
        canvas.begin_path()
        canvas.move_to(last_x, last_y)
        canvas.line_to(x, y)
        canvas.stroke()
        last_x, last_y = x, y

def on_mouse_up(x, y):
    global drawing
    drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

# Buttons
def clear_canvas(btn):
    canvas.fill_style = "black"
    canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    canvas.stroke_style = "white"
    with output:
        clear_output()
        print("Canvas cleared!")

def recognize(btn):
    # Get canvas data
    import cv2
    data = canvas.get_image_data(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    img_array = np.array(data).reshape(CANVAS_SIZE, CANVAS_SIZE, 4)
    gray = img_array[:, :, 0]  # Just use red channel (white strokes)
    
    # Predict
    results = predict(gray, model, config, device)
    
    with output:
        clear_output()
        print("\n" + "="*40)
        print("PREDICTIONS")
        print("="*40)
        for i, (name, prob) in enumerate(results):
            bar = "█" * int(prob * 30)
            print(f"{i+1}. {name.replace('_', ' '):<25} {prob*100:5.1f}% {bar}")
        print("="*40)

clear_btn = Button(description="Clear", button_style="warning")
clear_btn.on_click(clear_canvas)

recognize_btn = Button(description="Recognize", button_style="success")
recognize_btn.on_click(recognize)

# Model selector
def change_model(change):
    global model, device, config, current_model_key
    current_model_key = change['new']
    with output:
        clear_output()
        model, device, config = load_model(current_model_key)

model_dropdown = Dropdown(
    options=[('Old (MobileNetV3 64x64)', 'old'), ('New (EfficientNet-B2 96x96)', 'new')],
    value=current_model_key,
    description='Model:'
)
model_dropdown.observe(change_model, names='value')

# Layout
title = HTML("<h2>🎨 Quick Draw Demo</h2><p>Draw on the canvas and click Recognize!</p>")
buttons = HBox([recognize_btn, clear_btn, model_dropdown])
display(VBox([title, canvas, buttons, output]))

In [ ]:
# ============================================
# PRINT ALL CLASSES
# ============================================

print("All 340 Quick Draw Classes:")
print("=" * 80)
for i in range(0, len(CLASSES), 5):
    row = CLASSES[i:i+5]
    print("  ".join(f"{c:<15}" for c in row))
print("=" * 80)